In [2]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm
import plotly.graph_objects as go


def exp( x , A , tau ):
    return expon.cdf(x , A , tau) 

def gauss( x , N , mu , sigma):
    return N*norm.cdf(x , mu , sigma)

def exp_gauss_cdf(N_exp , A , tau):
    def fixed_exp( x , N , mu , sigma):
        return exp( x , N_exp , A , tau) + gauss( x , N , mu , sigma)
    return fixed_exp
    
def gauss_exp_cdf(N_g , mu , sigma):
    def fixed_gauss( x , N , A , tau):
        return exp( x , N , A , tau) + gauss( x , N_g , mu , sigma)
    return fixed_gauss

def exp_fit(cdf, A , tau, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, A = A , tau = tau )
    # n.fixed['N'] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

In [ ]:
# data_1 = np.genfromtxt("Data/timestamp/16_1_2026_15_48.csv", delimiter=',')
# data_2 = np.genfromtxt("Data/timestamp/16_1_2026_16_55.csv", delimiter=',')
data_3 = np.genfromtxt("Data/timestamp/16_1_2026_17_59.csv", delimiter=',')

def _clean(data, label):
    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]  # drop NaN/Inf
    if data.size == 0:
        raise ValueError(f"{label} has no finite entries; check the source file.")
    return data

# data_1 = _clean(data_1, "Dataset 1")
# data_2 = _clean(data_2, "Dataset 2")
data_3 = _clean(data_3, "Dataset 3")

n_bins = 50
# count_1, edges_1 = np.histogram(data_1, bins=n_bins , density=True)
# count_2, edges_2 = np.histogram(data_2, bins=n_bins , density=True)
count_3, edges_3 = np.histogram(data_3, bins=n_bins , density=True)

fig = go.Figure()

# fig.add_trace(go.Bar(x=edges_1[:-1], y=count_1, name='Histogram', width=np.diff(edges_1)))
# fig.add_trace(go.Bar(x=edges_2[:-1], y=count_2, name='Histogram 2', width=np.diff(edges_2)))
fig.add_trace(go.Bar(x=edges_3[:-1], y=count_3, name='Histogram 3', width=np.diff(edges_3)))

fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

ValueError: Dataset 2 has no finite entries; check the source file.

In [ ]:
k = exp_fit( exp , 5, 2e-6 , count_1 , edges_1)
display(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.599e+08 (χ²/ndof = 3330637.2)│              Nfcn = 657              │
│ EDM = nan (Goal: 0.0002)         │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│         INVALID Minimum          │   ABOVE EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ A    │   1.28    │    nan    │            │            │         │         │       │
│ 1 │ tau  │   2e-6    │    nan    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────┐
│     │   A tau │
├─────┼─────────┤
│   A │ nan nan │
│ tau │ nan nan │
└─────┴─────────┘